# PROJECT STAGE

Current stage: Research exploration

Code may later be migrated to src modules once stabilized.

# 1. Metadata

In [ ]:
# ============================================================
# NOTEBOOK METADATA
# ============================================================

NOTEBOOK_NAME = "02_asset_characterization_Momentum"
AUTHOR = "Juan Manuel Giacame"
CREATED = "2026-03-08"
UPDATED = "2026-03-12"
PROJECT = "macro-market-analysis"
STAGE = "Research"
VERSION = "0.1.0"
GIT_HASH = ""  # completar con commit hash si se usa git
EXPERIMENT_ID = f"{NOTEBOOK_NAME}_{CREATED}_{VERSION}"

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
PURPOSE = """
Compute momentum features for each asset in the universe.

This notebook calculates time-series momentum metrics using
rolling return aggregation across multiple horizons.

The resulting features are stored per asset in:

data/features/asset/Momentum/{asset}.parquet
"""

# ------------------------------------------------------------
# INPUT DATASETS
# ------------------------------------------------------------
INPUT_DATASETS = [
    "data/processed/prices.parquet"
]

# ------------------------------------------------------------
# OUTPUT DATASETS
# ------------------------------------------------------------
OUTPUT_DATASETS = [
    "data/features/asset/Momentum/{asset}.parquet"
]

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Momentum"

# ------------------------------------------------------------
# FEATURE DESCRIPTION
# ------------------------------------------------------------
FEATURE_DESCRIPTION = """
Momentum features measure the cumulative return of an asset
over multiple historical horizons.

These features capture trend persistence and are commonly used
in time-series momentum and cross-sectional momentum strategies.
"""

# ------------------------------------------------------------
# ASSETS UNIVERSE
# ------------------------------------------------------------
ASSETS_UNIVERSE = "All assets in data/processed/prices.parquet"

# ------------------------------------------------------------
# DEPENDENCIES
# ------------------------------------------------------------
DEPENDENCIES = ["pandas >= 2.0", "numpy", "plotly >= 5.0", "pathlib"]

# ------------------------------------------------------------
# NOTES
# ------------------------------------------------------------
NOTES = """
Features are computed independently per asset.

Cross-sectional transformations such as:
    - rankings
    - z-scores
    - dispersion
are handled later in:

03_systemic_features.ipynb
"""

# 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================

# -----------------------------
# Standard library
# -----------------------------
from pathlib import Path

# -----------------------------
# Data manipulation
# -----------------------------
import pandas as pd  # pandas >= 2.0
import numpy as np   # numpy >= 1.25

# Optional: display settings for research notebooks
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.options.mode.chained_assignment = None  # avoid SettingWithCopyWarning

# -----------------------------
# Parquet engine
# -----------------------------
import pyarrow  # pyarrow >= 12.0

# -----------------------------
# Visualization
# -----------------------------
import plotly.express as px  # plotly >= 5.0
import plotly.graph_objects as go

# -----------------------------
# Optional future imports from src/ once implemented
# -----------------------------
# from src.utils import panel_utils, plotting_utils

# 3. Configuration

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# DATA PATHS
# ------------------------------------------------------------
DATA_DIR = Path("../../data")
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"

# Input dataset for momentum calculations
PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"

# ------------------------------------------------------------
# FEATURE FAMILY
# ------------------------------------------------------------
FEATURE_FAMILY = "Momentum"

# ------------------------------------------------------------
# FEATURE PARAMETERS
# ------------------------------------------------------------
LOOKBACK_WINDOWS = [21, 63, 126, 252]

# Smoothing windows for velocity/acceleration derivatives (VEL_S, ACC_S)
SMOOTH_WINDOWS = {
    21: 3,
    63: 3,
    126: 5,
    252: 5
}

PERCENTILE_WINDOW = 252

# ------------------------------------------------------------
# UNIVERSE CONFIGURATION
# ------------------------------------------------------------
# Will be dynamically extracted from df_prices after loading
ASSETS = None
UNIVERSE_SOURCE = "prices_columns"

# ------------------------------------------------------------
# OUTPUT CONFIGURATION
# ------------------------------------------------------------
EXPORT_FORMAT = "parquet"
OVERWRITE_EXISTING = True

# ------------------------------------------------------------
# RESEARCH OPTIONS
# ------------------------------------------------------------
# Sample assets for visualizations and sanity checks
SAMPLE_ASSETS_FOR_PLOTS = [
    "BTC",
    "SPY"
]

# ------------------------------------------------------------
# RANDOM SEED
# ------------------------------------------------------------
RANDOM_SEED = 42

# 4. Data Loading

In [ ]:
# ============================================================
# 4. DATA LOADING
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Helper function: detect project root by searching for a marker folder
# ------------------------------------------------------------
def find_project_root(marker="data") -> Path:
    """
    Finds the first parent directory containing marker and returns its Path.
    """
    current_path = Path().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError(f"Project root with '{marker}/' directory not found.")

project_root = find_project_root("data")
print("Project root detected at:", project_root)

# ------------------------------------------------------------
# Define data paths
# ------------------------------------------------------------
DATA_DIR = project_root / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features" / "asset" / "volatility"

PRICES_PATH = PROCESSED_DATA_DIR / "prices.parquet"
RETURNS_PATH = PROCESSED_DATA_DIR / "returns.parquet"

# ------------------------------------------------------------
# Verify files exist
# ------------------------------------------------------------
for path in [PRICES_PATH, RETURNS_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found at {path}")
print("All input datasets found ✔")

# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------
df_prices = pd.read_parquet(PRICES_PATH)
df_returns = pd.read_parquet(RETURNS_PATH)

# Ensure datetime index
if "date" in df_prices.columns:
    df_prices["date"] = pd.to_datetime(df_prices["date"])
    df_prices.set_index("date", inplace=True)
df_prices.sort_index(inplace=True)

if "date" in df_returns.columns:
    df_returns["date"] = pd.to_datetime(df_returns["date"])
    df_returns.set_index("date", inplace=True)
df_returns.sort_index(inplace=True)

print(f"Prices dataset shape: {df_prices.shape}")
print(f"Returns dataset shape: {df_returns.shape}")

# ------------------------------------------------------------
# Infer asset universe
# ------------------------------------------------------------
ASSETS = df_prices.columns.tolist()
print(f"\nNumber of assets in universe: {len(ASSETS)}")
print("Sample assets:", ASSETS[:10])

# ------------------------------------------------------------
# Quick inspection
# ------------------------------------------------------------
display(df_prices.head())
display(df_returns.head())

# 5. Core Computation
MOMENTUM

Momentum represents the persistence of price movements over time.
Assets exhibiting positive momentum tend to continue outperforming,
while negative momentum assets tend to underperform.

Trend analysis allows us to quantify:

- Direction of movement
- Strength of persistence
- Stability of trends
- Speed of price evolution

Momentum features are fundamental inputs for both
asset allocation and alpha generation models.

In this section we construct statistical measures that
capture trend behavior across multiple horizons.

### 5.1 Multi-Horizon Momentum

Momentum measures the persistence of price trends over time.

Instead of relying on a single lookback window, we compute momentum
across multiple horizons to capture short-, medium-, and long-term trends.

This approach allows us to characterize how trend strength evolves
across different investment horizons.

Momentum is computed using cumulative log returns:

$$ Momentum(t, n) = log(P_t / P_{t-n}) $$

Where:

$ - P_t$ = current price 
$ - P_{t-n}$ = price n periods ago 
$ - n$ = lookback window 

We compute momentum over multiple horizons:

- 21 days  → Short-term momentum (~1 month)
- 63 days  → Medium-term momentum (~3 months)
- 126 days → Intermediate trend (~6 months)
- 252 days → Long-term momentum (~1 year)

Multi-horizon momentum helps identify trend persistence
and regime transitions across assets.

Structural NaNs naturally appear during the warmup period and are preserved.
These observations will later be excluded when defining the research dataset.

In [ ]:
# ============================================================
# 5. CORE COMPUTATIONS
# Feature Block: Momentum Level (MOM_h)
# ============================================================

# ------------------------------------------------------------
# Compute log momentum for each horizon
# ------------------------------------------------------------
momentum_level = {
    f"MOM_{h}": np.log(df_prices / df_prices.shift(h))
    for h in LOOKBACK_WINDOWS
}

# ------------------------------------------------------------
# Concatenate into a single DataFrame
# MultiIndex columns: feature x asset
# ------------------------------------------------------------
momentum_level = pd.concat(momentum_level, axis=1)
momentum_level.columns.names = ["feature", "asset"]

# Ensure consistent column ordering
momentum_level = momentum_level.sort_index(axis=1)

# ------------------------------------------------------------
# Note on NaNs
# ------------------------------------------------------------
# NaNs are expected due to lookback windows (warmup period)

# ------------------------------------------------------------
# Quick inspection
# ------------------------------------------------------------
print("Momentum Level features created:")
print(momentum_level.columns)

print("Dataset shape:", momentum_level.shape)

momentum_level.head()

### 5.2 Momentum Stability

Momentum alone does not indicate whether a trend is reliable.

Momentum Stability measures how consistent momentum remains
through time. Stable momentum suggests persistent trends,
while unstable momentum indicates noisy or fragile price movements.

We estimate momentum stability as the rolling standard deviation
of momentum values.

Lower dispersion implies stronger trend persistence.

Mathematically:

$$ Momentum Stability = std(Momentum_t) over rolling window $$

Where:

- Low values → stable trend
- High values → unstable or reversing trend

This metric helps distinguish between sustainable trends
and short-lived market moves.

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Feature Block: Momentum Stability (MOM_h_STAB)
# ==========================================

# Dictionary to store stability features
momentum_stab = {}

# Loop over each momentum horizon
for h in LOOKBACK_WINDOWS:
    
    feature_name = f"MOM_{h}_STAB"
    
    # Rolling standard deviation of MOM_h
    stab = momentum_level[f"MOM_{h}"].rolling(h).std()
    
    # Store result
    momentum_stab[feature_name] = stab


# Concatenate into MultiIndex DataFrame
momentum_stab = pd.concat(momentum_stab, axis=1)
momentum_stab.columns.names = ["feature", "asset"]

# Sort columns for consistent ordering
momentum_stab.sort_index(axis=1, inplace=True)


# Quick inspection
print("Momentum Stability features created:")
print(momentum_stab.columns)

momentum_stab.head()

### 5.3 Momentum Z-Score (Normalized Trend Strength)

Raw momentum values are not directly comparable across assets,
as each asset exhibits different volatility and trend magnitude.

To normalize trend strength, momentum is standardized using
a rolling Z-score.

Z-score measures how extreme the current momentum is relative
to its historical distribution.

Mathematically:

$$ Z_t = (MOM_t - μ_t) / σ_t "" $$

Where:

$ - MOM_t $ : momentum at time t 
$ - μ_t $  : rolling mean of momentum 
$ - σ_t $ : rolling standard deviation 

Interpretation:

- Z > 2  → unusually strong trend
- Z ≈ 0  → normal trend conditions
- Z < -2 → unusually weak or bearish trend

This transformation allows comparison of trend strength
across assets and time.

In [ ]:
# ============================================================
# 5. CORE COMPUTATIONS
# Feature Block: Standardized Momentum (MOM_h_Z)
# ============================================================

# Rolling window for z-score normalization (1 year)
Z_WINDOW = 252

momentum_z = {}

# ------------------------------------------------------------
# Compute rolling z-score of momentum
# ------------------------------------------------------------
for h in LOOKBACK_WINDOWS:

    mom = momentum_level[f"MOM_{h}"]

    rolling_mean = mom.rolling(Z_WINDOW).mean()
    rolling_std  = mom.rolling(Z_WINDOW).std()

    momentum_z[f"MOM_{h}_Z"] = (mom - rolling_mean) / rolling_std

# ------------------------------------------------------------
# Concatenate into single DataFrame
# ------------------------------------------------------------
momentum_z = pd.concat(momentum_z, axis=1)
momentum_z.columns.names = ["feature", "asset"]

# Consistent ordering
momentum_z = momentum_z.sort_index(axis=1)

# ------------------------------------------------------------
# NaNs are expected due to rolling windows
# ------------------------------------------------------------

# Quick inspection
print(f"Standardized Momentum features (rolling {Z_WINDOW}d) created:")
print(momentum_z.columns)
print("Dataset shape:", momentum_z.shape)

momentum_z.head()

#### 5.3.1 Momentum Percentile

The **Momentum Percentile** measures the relative position of the current momentum value within its own historical distribution.

For each asset and lookback window, we compute the rolling percentile rank of the momentum level over a specified historical window.

This transformation provides a **distribution-free normalization** of momentum, allowing us to identify whether the current trend is weak, neutral, or extreme relative to the asset’s own history.

Mathematically: $$ MOM_{w,PCTL,t}=\frac{rank\left(MOM_{w,t} \;|\; MOM_{w,t-L+1}, ..., MOM_{w,t}\right)}{L} $$

Where:

- $w$ = momentum lookback window (e.g. 21, 63, 126, 252)
- $L$ = percentile window length
- $rank(\cdot)$ = rank of the current value within the rolling window

Interpretation:

| Percentile | Interpretation |
|---|---|
| ~0.0 – 0.2 | Very weak momentum |
| ~0.4 – 0.6 | Neutral momentum |
| ~0.8 – 1.0 | Strong / extreme momentum |

Unlike Z-scores, percentile normalization does **not assume a specific distribution**, making it more robust to outliers and heavy-tailed return behavior.

These features are labeled:

```
MOM_21_PCTL
MOM_63_PCTL
MOM_126_PCTL
MOM_252_PCTL
```

and represent **momentum regimes relative to each asset's own historical behavior**.

In [ ]:
# ============================================================
# Momentum Percentile (fully vectorized)
# ============================================================

mperc_features = {}

window = PERCENTILE_WINDOW

for mom_window in LOOKBACK_WINDOWS:

    mom_label = f"MOM_{mom_window}"
    perc_label = f"MOM_{mom_window}_PCTL"

    mom_df = momentum_level[mom_label]

    percentile_df = mom_df.rolling(window).rank(pct=True)

    mperc_features[perc_label] = percentile_df


momentum_percentile = pd.concat(mperc_features, axis=1)
momentum_percentile.columns.names = ["feature", "asset"]

print("Momentum percentile shape:", momentum_percentile.shape)

display(momentum_percentile.tail())

### 5.4 Momentum Agreement Index

Momentum signals across different horizons may diverge.

The Momentum Agreement Index measures whether short-, medium-,
and long-term momentum signals point in the same direction.

Trend conviction increases when multiple horizons agree.

We define agreement as the average sign of momentum signals:

$$ Agreement = mean(sign(Momentum_i)) $$

Where Momentum_i represents momentum computed
across multiple lookback windows.

Interpretation:

- +1 → Strong bullish agreement
- 0  → Mixed or transition regime
- -1 → Strong bearish agreement

This metric helps detect high-conviction trends
and potential regime shifts.

In [ ]:
# ==========================================
# Feature Block: Momentum Alignment (cross-horizon agreement)
# ==========================================

assets = momentum_level.columns.levels[1]
alignment_horizons = [f"MOM_{h}" for h in LOOKBACK_WINDOWS]

# Dictionary to store alignment per asset
momentum_align_dict = {}

for asset in assets:
    
    # Subset momentum features for the current asset
    df_asset = momentum_level.xs(asset, level="asset", axis=1)
    
    # Compute sign of momentum for each horizon
    momentum_sign = df_asset[alignment_horizons].apply(np.sign)
    
    # Average sign across horizons (agreement index)
    momentum_align_dict[asset] = momentum_sign.mean(axis=1)

# Concatenate results across assets
momentum_align = pd.concat(momentum_align_dict, axis=1)

# Recreate MultiIndex structure
momentum_align.columns = pd.MultiIndex.from_product(
    [["MOM_ALIGN"], assets]
)
momentum_align.columns.names = ["feature", "asset"]

# Sort columns for consistency
momentum_align.sort_index(axis=1, inplace=True)

print("Momentum Alignment feature created:")
print(momentum_align.columns)

## 5.4.1 Momentum Agreement Index (Z-Score Based)

Objective

Measure multi-horizon trend alignment using standardized momentum (Z-scores) 
instead of raw momentum values.

While raw momentum agreement captures directional alignment, it treats weak 
and strong signals equally. Using Z-scores allows us to account for statistical 
significance relative to historical behavior.

---

Economic Intuition

Markets exhibit stronger regime behavior when multiple time horizons 
show statistically significant trend alignment.

If:

- Short-term momentum is positive,
- Medium-term momentum is positive,
- Long-term momentum is positive,

and all are meaningfully different from zero (high |Z|),

then the probability of a persistent bullish regime increases.

The same logic applies for bearish alignment.

---

Construction

For each asset and date:

1. Compute the sign of standardized momentum (Z-score).
2. Average the signs across all horizons.
3. Optionally filter weak signals (|Z| below threshold).

The resulting index lies in:

- +1  → all horizons significantly bullish
-  0  → mixed or weak signals
- -1  → all horizons significantly bearish

---

Interpretation

This feature represents a:

Multi-Scale Significant Trend Alignment Score

It can later be used for:
- Regime detection
- Signal filtering
- Conditional allocation rules

In [ ]:
# ==========================================
# Momentum Agreement Index (Z-Score Filtered)
# ==========================================

# Available assets
assets = momentum_z.columns.levels[1]

# Z-score momentum features
alignment_horizons = momentum_z.columns.levels[0]

# Z-score significance threshold
Z_THRESHOLD = 0.5

# Dictionary to store alignment per asset
momentum_align_dict = {}

for asset in assets:
    
    # Select z-scored momentum for the asset
    df_asset = momentum_z.xs(asset, level="asset", axis=1)
    
    # Extract Z-score horizons
    z_values = df_asset[alignment_horizons]
    
    # Filter weak signals while preserving NaNs
    z_filtered = z_values.mask(np.abs(z_values) < Z_THRESHOLD)
    
    # Compute sign of filtered signals
    z_sign = np.sign(z_filtered)
    
    # Average sign across horizons
    momentum_align_dict[asset] = z_sign.mean(axis=1)

# ------------------------------------------
# Build final DataFrame
# ------------------------------------------

momentum_align_z = pd.concat(momentum_align_dict, axis=1)

momentum_align_z.columns = pd.MultiIndex.from_product(
    [["MOM_ALIGN_Z"], assets]
)

momentum_align_z.columns.names = ["feature", "asset"]

momentum_align_z.sort_index(axis=1, inplace=True)

print("Filtered Momentum Agreement Index created successfully:")
print(momentum_align_z.head())

In [ ]:
# ------------------------------------------
# Structure checks
# ------------------------------------------

print("Shape:")
print(momentum_align.shape)

print("\nColumn levels:")
print(momentum_align.columns.names)

print("\nFeature names:")
print(momentum_align.columns.levels[0])

print("\nAssets:")
print(momentum_align.columns.levels[1][:10])

print("\nIndex sample:")
print(momentum_align.index[:5])

print("\nHead:")
print(momentum_align.tail())

In [ ]:
momentum_align.stack().hist(bins=50)

In [ ]:
momentum_align.describe()

In [ ]:
# ==========================================
# Cross-Sectional Momentum Rank
# ==========================================

# Compute cross-sectional percentile rank per horizon

momentum_cs = (
    momentum_level
        .T
        .groupby(level="feature")
        .rank(pct=True)
        .T
)

# Rename features to indicate cross-sectional ranking
momentum_cs.columns = pd.MultiIndex.from_tuples(
    [(f"{feat}_CS", asset) for feat, asset in momentum_cs.columns],
    names=momentum_cs.columns.names
)

# Ensure consistent column ordering
momentum_cs.sort_index(axis=1, inplace=True)

print("Cross-sectional momentum rank created successfully:")
momentum_cs.tail()

In [ ]:
momentum_cs.describe()

In [ ]:
# ==========================================
# Cross-Sectional Momentum Z-Score
# ==========================================

momentum_cs_z = (
    momentum_z
        .T
        .groupby(level="feature")
        .transform(lambda x: (x - x.mean()) / x.std())
        .T
)

# Rename features
momentum_cs_z.columns = pd.MultiIndex.from_tuples(
    [(feat.replace("_Z", "_CS_Z"), asset) for feat, asset in momentum_cs_z.columns],
    names=momentum_cs_z.columns.names
)

# Ensure consistent ordering
momentum_cs_z.sort_index(axis=1, inplace=True)

momentum_cs_z.tail()

## 5.5 Momentum Dynamics


In [ ]:
# ==========================================
# Momentum Velocity
# ==========================================

# Compute velocity as first difference of momentum_level
momentum_vel = momentum_level.diff()

# Rename columns to keep MultiIndex convention
momentum_vel.columns = pd.MultiIndex.from_tuples(
    [(f"{feat}_VEL", asset) for feat, asset in momentum_vel.columns],
    names=momentum_vel.columns.names
)

# Quick inspection
print("Momentum Velocity features created:")
print(momentum_vel.columns)
momentum_vel.head()


# ==========================================
# Momentum Acceleration
# ==========================================

# Compute acceleration as difference of velocity
momentum_acc = momentum_vel.diff()

# Rename columns carefully: replace _VEL with _ACC
momentum_acc.columns = pd.MultiIndex.from_tuples(
    [(feat.replace("_VEL", "_ACC"), asset) for feat, asset in momentum_acc.columns],
    names=momentum_acc.columns.names
)
#
# NANs are espected due to Warm up

# Quick inspection
print("Momentum Acceleration features created:")
print(momentum_acc.columns)
momentum_acc.head()

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Feature Block: Dynamics (variable smoothing)
# ==========================================

# Dictionaries to store smoothed features
momentum_vel_s = {}
momentum_acc_s = {}

# Loop over each momentum horizon
for h in LOOKBACK_WINDOWS:
    
    # Retrieve smoothing window from configuration
    smooth_w = SMOOTH_WINDOWS[h]
    
    # Feature names
    vel_s_name = f"MOM_{h}_VEL_S"
    acc_s_name = f"MOM_{h}_ACC_S"
    
    # Retrieve raw velocity and acceleration
    vel_raw = momentum_vel[f"MOM_{h}_VEL"]
    acc_raw = momentum_acc[f"MOM_{h}_ACC"]
    
    # Apply variable rolling smoothing
    vel_s = vel_raw.rolling(smooth_w).mean()
    acc_s = acc_raw.rolling(smooth_w).mean()
    
    # Store results
    momentum_vel_s[vel_s_name] = vel_s
    momentum_acc_s[acc_s_name] = acc_s


# Concatenate into MultiIndex DataFrames
momentum_vel_s = pd.concat(momentum_vel_s, axis=1)
momentum_acc_s = pd.concat(momentum_acc_s, axis=1)

# Clean and organize columns
for df in [momentum_vel_s, momentum_acc_s]:
    
    # Set MultiIndex column names
    df.columns.names = ["feature", "asset"]
    
    # Sort columns for readability
    df.sort_index(axis=1, inplace=True)
    



# Quick inspection
print("Momentum Dynamics (variable smoothing) features created:")
print(momentum_vel_s.columns)
print(momentum_acc_s.columns)

momentum_vel_s.head()

In [ ]:
# ==========================================
# Momentum Dynamics Consistency Check con %
# ==========================================

consistency_check = {}

for h in LOOKBACK_WINDOWS:
    
    # Raw vs smoothed
    vel_raw  = momentum_vel[f"MOM_{h}_VEL"]
    vel_s    = momentum_vel_s[f"MOM_{h}_VEL_S"]
    
    acc_raw  = momentum_acc[f"MOM_{h}_ACC"]
    acc_s    = momentum_acc_s[f"MOM_{h}_ACC_S"]
    
    # Alinear índices y columnas
    vel_raw_aligned, vel_s_aligned = vel_raw.align(vel_s, join="inner")
    acc_raw_aligned, acc_s_aligned = acc_raw.align(acc_s, join="inner")
    
    # Correlation por asset
    corr_vel = vel_raw_aligned.corrwith(vel_s_aligned)
    corr_acc = acc_raw_aligned.corrwith(acc_s_aligned)
    
    # Varianza de raw y smoothed
    var_vel_raw = vel_raw_aligned.var()
    var_vel_s   = vel_s_aligned.var()
    var_acc_raw = acc_raw_aligned.var()
    var_acc_s   = acc_s_aligned.var()
    
    # % Reducción de varianza
    vel_var_reduction_pct = ((var_vel_raw - var_vel_s) / var_vel_raw * 100).mean()
    acc_var_reduction_pct = ((var_acc_raw - var_acc_s) / var_acc_raw * 100).mean()
    
    # Sign flips
    sign_flips_vel = (np.sign(vel_raw_aligned) != np.sign(vel_s_aligned))
    sign_flips_acc = (np.sign(acc_raw_aligned) != np.sign(acc_s_aligned))
    
    # % de cambio de signo
    vel_sign_flips_pct = sign_flips_vel.sum().sum() / np.prod(sign_flips_vel.shape) * 100
    acc_sign_flips_pct = sign_flips_acc.sum().sum() / np.prod(sign_flips_acc.shape) * 100
    
    # Guardar resultados
    consistency_check[h] = {
        "vel_corr_mean": corr_vel.mean(),
        "vel_var_reduction_pct": vel_var_reduction_pct,
        "vel_sign_flips_pct": vel_sign_flips_pct,
        "acc_corr_mean": corr_acc.mean(),
        "acc_var_reduction_pct": acc_var_reduction_pct,
        "acc_sign_flips_pct": acc_sign_flips_pct
    }

# Convertir a DataFrame para inspección
consistency_df = pd.DataFrame(consistency_check).T
consistency_df

## 5.6 Momentum Strength Index (Weighted & Smoothed)

Individual momentum horizons capture trend dynamics at
different temporal scales. However, analyzing them
separately does not provide a unified structural view
of market strength.

To aggregate cross-horizon information, we define the
Momentum Strength Index (MSI) as a weighted combination
of normalized momentum signals.

Let $ Z_t(h) $ denote the Z-scored momentum for horizon h.

The weighted MSI is defined as:

$$ MSI_t = Σ_h ( w_h * Z_t(h) ) $$

where weights satisfy:

$$ Σ_h w_h = 1 $$

The weighting scheme reflects structural importance
across horizons.

In our baseline specification:

w_21  = 0.1  
w_63  = 0.2  
w_126 = 0.3  
w_252 = 0.4  

Longer horizons receive higher weights to emphasize
structural regime components over tactical noise.

Since MSI aggregates multiple horizons, it is further
smoothed to enhance regime stability:

$$ M̃SI_t = (1 / k) * Σ_{i=0}^{k-1} MSI_{t-i} $$

with k = 5.

This produces a structurally stable, cross-horizon
trend-strength indicator suitable for regime detection
and allocation modeling.

In [ ]:
# ==========================================
# Momentum Strength Index (MSI)
# ==========================================

weights = {
    "MOM_21_Z": 0.1,
    "MOM_63_Z": 0.2,
    "MOM_126_Z": 0.3,
    "MOM_252_Z": 0.4,
}

msi_components = []

for feat, w in weights.items():
    
    # Select Z-scored momentum for the horizon
    df_feat = momentum_z.xs(feat, level="feature", axis=1)
    
    # Apply weight
    df_weighted = df_feat * w
    
    msi_components.append(df_weighted)

# Aggregate weighted horizons
msi_raw = pd.concat(msi_components).groupby(level=0).sum()

# Restore MultiIndex structure
msi_raw.columns = pd.MultiIndex.from_product(
    [["MSI"], msi_raw.columns],
    names=momentum_z.columns.names
)

# ------------------------------------------
# Smooth the index
# ------------------------------------------

msi_smooth = msi_raw.rolling(window=5).mean()

msi_smooth.columns = pd.MultiIndex.from_product(
    [["MSI_S"], msi_smooth.columns.get_level_values("asset")],
    names=msi_raw.columns.names
)

print("Momentum Strength Index created successfully:")
msi_smooth.tail()

In [ ]:
msi_smooth.stack().hist(bins=50)

## 5.6.1 MSI Velocity (Smoothed)

To capture structural changes in aggregated trend strength,
we compute the first derivative of the smoothed MSI.

MSI velocity is defined as:

$$ MSI_{Vel_t} = M̃SI_t - M̃SI_{t-1} $$

This measures the rate of change in structural momentum
across horizons.

Because MSI is already smoothed, velocity reflects
structural acceleration rather than tactical noise.

To ensure stability, MSI velocity is additionally
smoothed using a 5-day rolling mean:

$$ M̃SI_{Vel_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Vel_{t-i}} $$

This produces a stable structural force indicator,
useful for identifying regime transitions.

In [ ]:
# MSI velocity
msi_velocity = msi_smooth.diff()

# Smoothed velocity
msi_velocity_smooth = (
    msi_velocity
        .rolling(window=5)
        .mean()
)

# Asignar MultiIndex consistente
msi_velocity_smooth.columns = pd.MultiIndex.from_arrays(
    [
        ["MSI_VEL_S"] * len(msi_velocity_smooth.columns),
        msi_velocity_smooth.columns.get_level_values("asset")
    ],
    names=msi_velocity_smooth.columns.names
)

msi_velocity_smooth.columns

## 5.6.2 MSI Acceleration (Smoothed)

To detect structural inflection points, we compute
the second derivative of the smoothed MSI.

MSI acceleration is defined as:

$$ MSI_{Acc_t} = MSI_{Vel_t} - MSI_{Vel_{t-1}} $$

Equivalently:

$$ MSI_{Acc_t} = Δ² M̃SI_t $$

Acceleration captures changes in structural momentum force,
identifying reinforcement or exhaustion of dominant regimes.

Given the noise amplification inherent in second derivatives,
MSI acceleration is smoothed using a 5-day rolling mean:

$$ M̃SI_{Acc_t} = (1 / 5) * Σ_{i=0}^{4} MSI_{Acc_{t-i}} $$

The resulting signal provides a stable structural
inflection indicator suitable for regime-state modeling.

In [ ]:
msi_acceleration = msi_velocity.diff()

msi_acceleration_smooth = (
    msi_acceleration
        .rolling(window=5)
        .mean()
)

msi_acceleration_smooth.columns = pd.MultiIndex.from_product(
    [["MSI_ACC_S"], msi_acceleration.columns.get_level_values("asset")],
    names=msi_acceleration.columns.names
)

msi_acceleration_smooth.columns






In [ ]:
msi_acceleration = msi_velocity.diff()

msi_acceleration_smooth = (
    msi_acceleration
        .rolling(window=5)
        .mean()
)

msi_acceleration_smooth.columns = pd.MultiIndex.from_arrays(
    [
        ["MSI_ACC_S"] * len(msi_acceleration_smooth.columns),
        msi_acceleration_smooth.columns.get_level_values("asset")
    ],
    names=msi_acceleration_smooth.columns.names
)

msi_acceleration_smooth.columns

In [ ]:
momentum_structural = pd.concat([
    msi_raw,
    msi_smooth,
    msi_velocity_smooth,
    msi_acceleration_smooth
], axis=1).sort_index(axis=1)

momentum_structural.columns

In [ ]:
asset = "BTC"  # ajustar si es necesario

msi_r = msi_raw.xs("MSI", level="feature", axis=1)[asset]
msi_s = msi_smooth.xs("MSI_S", level="feature", axis=1)[asset]

# Align
df = pd.concat([msi_r, msi_s], axis=1).dropna()
df.columns = ["raw", "smooth"]

# Correlation
corr = df["raw"].corr(df["smooth"])

# Variance reduction
std_raw = df["raw"].std()
std_smooth = df["smooth"].std()
var_reduction = 1 - (std_smooth / std_raw)

# Sign flips
sign_flips_raw = (df["raw"].diff().apply(np.sign).diff() != 0).sum()
sign_flips_smooth = (df["smooth"].diff().apply(np.sign).diff() != 0).sum()
flip_reduction = 1 - (sign_flips_smooth / sign_flips_raw)

print("Correlation:", round(corr,4))
print("Std raw:", std_raw)
print("Std smooth:", std_smooth)
print("Variance reduction (%):", round(var_reduction*100,2))
print("Sign flips raw:", sign_flips_raw)
print("Sign flips smooth:", sign_flips_smooth)
print("Flip reduction (%):", round(flip_reduction*100,2))

In [ ]:
vel_r = msi_velocity.xs("MSI_S", level="feature", axis=1)[asset]
vel_s = msi_velocity_smooth.xs("MSI_VEL_S", level="feature", axis=1)[asset]

df_vel = pd.concat([vel_r, vel_s], axis=1).dropna()
df_vel.columns = ["raw", "smooth"]

corr_v = df_vel["raw"].corr(df_vel["smooth"])
std_raw_v = df_vel["raw"].std()
std_smooth_v = df_vel["smooth"].std()
var_red_v = 1 - (std_smooth_v / std_raw_v)

sign_flips_raw_v = (df_vel["raw"].diff().apply(np.sign).diff() != 0).sum()
sign_flips_smooth_v = (df_vel["smooth"].diff().apply(np.sign).diff() != 0).sum()
flip_red_v = 1 - (sign_flips_smooth_v / sign_flips_raw_v)

print("Velocity Correlation:", round(corr_v,4))
print("Velocity Variance reduction (%):", round(var_red_v*100,2))
print("Velocity Flip reduction (%):", round(flip_red_v*100,2))

In [ ]:
sorted(set(momentum_structural.columns.get_level_values("feature")))

## 5.7 Concatenation 

In [ ]:
# ==========================================
# 5. CORE COMPUTATIONS
# Concatenación final de todas las features de Momentum
# ==========================================

# Lista de DataFrames por bloques ya calculados
momentum_blocks = [
    momentum_level,       # MOM_h
    momentum_z,           # MOM_h_Z
    momentum_vel,         # MOM_h_VEL
    momentum_vel_s,       # MOM_h_VEL_S
    momentum_acc,         # MOM_h_ACC
    momentum_acc_s,       # MOM_h_ACC_S
    momentum_stab,        # MOM_h_STAB
    momentum_align,       # MOM_ALIGN
    momentum_align_z,     # MOM_ALIGN_Z
    momentum_structural,  # MSI_RAW, MSI_S, MSI_VEL_S, MSI_ACC_S
    momentum_percentile
    

]

# ------------------------------------------
# Concatenación
# ------------------------------------------

momentum_final = pd.concat(momentum_blocks, axis=1)

# ------------------------------------------
# Validación estructura MultiIndex
# ------------------------------------------

assert momentum_final.columns.nlevels == 2, "Columns must be a 2-level MultiIndex."

# Asegurar nombres consistentes
momentum_final.columns.names = ["feature", "asset"]

# ------------------------------------------
# Detectar columnas duplicadas
# ------------------------------------------

duplicated_cols = momentum_final.columns.duplicated()

if duplicated_cols.any():
    
    dup_list = momentum_final.columns[duplicated_cols]
    
    raise ValueError(
        f"Duplicate feature columns detected:\n{dup_list}"
    )

# ------------------------------------------
# Orden consistente (feature → asset)
# ------------------------------------------

momentum_final.sort_index(axis=1, inplace=True)

# ------------------------------------------
# Quick inspection
# ------------------------------------------

print("Momentum final concatenado correctamente.")
print()

print("Column levels:")
print(momentum_final.columns.names)

print()
print("Total features:", momentum_final.columns.get_level_values("feature").nunique())
print("Total assets:", momentum_final.columns.get_level_values("asset").nunique())

print()
print("Preview columns:")
print(momentum_final.columns[:10])

momentum_final.head()

In [ ]:
momentum_final.columns.get_level_values("feature").unique()

# 6. Diagnosis/Validation

## 6.1 Dataset integrity checks

In [ ]:
# ==========================================
# 6.1 DATASET INTEGRITY CHECKS
# ==========================================

features_panel = momentum_final

print("===== DATASET DIMENSIONS =====")

# Rows correspond to time observations (dates)
# Columns correspond to feature × asset combinations

print("Rows (dates):", features_panel.shape[0])
print("Columns (feature × asset):", features_panel.shape[1])

print()


print("===== MULTIINDEX STRUCTURE =====")

# Validate MultiIndex structure

print("Column levels:", features_panel.columns.names)

n_features = features_panel.columns.get_level_values("feature").nunique()
n_assets = features_panel.columns.get_level_values("asset").nunique()

print("Number of features:", n_features)
print("Number of assets:", n_assets)

print()


print("===== FEATURE LIST =====")

# List of computed features

features = features_panel.columns.get_level_values("feature").unique()

print(features)

print()


print("===== ASSET LIST =====")

assets = features_panel.columns.get_level_values("asset").unique()

print(assets)

print()


print("===== MISSING DATA CHECK =====")

# Rolling windows naturally generate NaNs at the beginning

nan_counts = features_panel.isna().sum().sum()
nan_ratio = features_panel.isna().mean().mean()

print("Total NaN values:", nan_counts)
print("Average NaN ratio:", round(nan_ratio, 4))

print()


print("===== DATE RANGE =====")

print("Start date:", features_panel.index.min())
print("End date:", features_panel.index.max())

print()


print("===== BASIC STATISTICS =====")

# Inspect summary statistics

display(features_panel.describe().T.head())

## 6.2 Feature Diagnostics

Beyond structural validation, we analyze the statistical behavior of the
constructed features.

These diagnostics help identify:

- degenerate signals (no cross-sectional variation)
- unstable distributions
- redundant features
- highly correlated signals

The following diagnostics are performed:

1. Cross-Sectional Dispersion
2. Cross-Sectional Distribution
3. Feature Correlation
4. Feature Clustering

### 6.2.1 Cross-Sectional Signal Dispersion

We evaluate whether each feature meaningfully differentiates assets at a given
point in time.

For each feature and date we compute the cross-sectional standard deviation
across assets.

Higher dispersion indicates stronger differentiation between assets,
which is desirable for cross-sectional models.

In [ ]:
# ==========================================
# 6.2.1 CROSS-SECTIONAL SIGNAL DISPERSION
# ==========================================

print("===== CROSS-SECTIONAL SIGNAL DISPERSION =====")

panel = features_panel

# Transpose for easier feature grouping
panel_t = panel.T  # (feature, asset) x date

# Compute dispersion across assets
cs_dispersion = panel_t.groupby(level="feature").std().T

# Average dispersion through time
avg_dispersion = cs_dispersion.mean().sort_values(ascending=False)

print("\nAverage cross-sectional dispersion per feature:")
print(avg_dispersion)

print("\nTop 10 most dispersed signals:")
print(avg_dispersion.head(10))

print("\nBottom 10 least dispersed signals:")
print(avg_dispersion.tail(10))

### 6.2.1b Normalized Signal Dispersion

To account for differences in signal scale, we compute normalized dispersion.

This metric compares cross-sectional dispersion to the average signal magnitude.

Higher values indicate stronger differentiation relative to signal amplitude.

In [ ]:
# ==========================================
# NORMALIZED SIGNAL DISPERSION
# ==========================================

print("\n===== NORMALIZED SIGNAL DISPERSION =====")

panel = features_panel
panel_t = panel.T

# Average signal magnitude
signal_scale = (
    panel_t
    .abs()
    .groupby(level="feature")
    .mean()
    .mean(axis=1)
)

# Normalized dispersion
normalized_dispersion = (avg_dispersion / signal_scale).sort_values(ascending=False)

print("\nNormalized dispersion per feature:")
print(normalized_dispersion)

print("\nTop 10 normalized signals:")
print(normalized_dispersion.head(10))

print("\nBottom 10 normalized signals:")
print(normalized_dispersion.tail(10))

### 6.2.2 Cross-Sectional Rank Spread

While cross-sectional dispersion measures the overall variability of a signal
across assets, it does not necessarily indicate whether the signal effectively
separates the highest and lowest ranked assets.

To evaluate the ranking power of each feature, we compute the **cross-sectional
rank spread**.

For each date, assets are ranked according to the signal value. We then
compare the average signal of the highest-ranked assets to the lowest-ranked
assets.

The cross-sectional rank spread is defined as:
 $Spread_{f,t} = E[x_{f,i,t} | i \in Top_q] - E[x_{f,i,t} | i \in Bottom_q]$.

where:

- \( f \) denotes the feature  
- \( i \) denotes the asset  
- \( t \) denotes the time index  
- \( q \) denotes the quantile threshold

In this study we use a **20% quantile threshold**, meaning that the top and
bottom 20% of assets are used to compute the spread.

Interpretation:

• **High spread**

The signal clearly differentiates between top and bottom assets and may be
useful for cross-sectional ranking strategies.

• **Moderate spread**

The signal provides some ranking information but may be weaker.

• **Low spread**

The signal does not meaningfully separate assets and may contain little
cross-sectional information.

In [ ]:
# ==========================================
# 6.2.2 CROSS-SECTIONAL RANK SPREAD
# ==========================================

print("\n===== CROSS-SECTIONAL RANK SPREAD =====")

panel = features_panel

base_q = 0.2

rank_spreads = {}

for feature in panel.columns.get_level_values("feature").unique():

    feature_df = panel[feature]  # date x asset

    n_assets = feature_df.shape[1]

    # number of assets to select
    n_select = max(1, int(np.floor(n_assets * base_q)))

    # effective quantile
    q = n_select / n_assets

    ranks = feature_df.rank(axis=1, pct=True)

    top = feature_df.where(ranks >= 1 - q)
    bottom = feature_df.where(ranks <= q)

    spread = top.mean(axis=1) - bottom.mean(axis=1)

    rank_spreads[feature] = spread.mean()

rank_spreads = pd.Series(rank_spreads).sort_values(ascending=False)
print(f"Assets: {n_assets} | Effective quantile: {q:.2f}")
print(rank_spreads)

In [ ]:
feature_diagnostics = pd.concat([
    avg_dispersion,
    normalized_dispersion,
    rank_spreads
], axis=1)

feature_diagnostics.columns = [
    "dispersion",
    "normalized_dispersion",
    "rank_spread"
]

feature_diagnostics.sort_values(
    "rank_spread",
    ascending=False
)

In [ ]:
fig = px.bar(
    avg_dispersion.sort_values(),
    orientation="h",
    title="Cross-Sectional Signal Dispersion",
    labels={"value": "Dispersion", "index": "Feature"}
)

fig.show()

In [ ]:
fig = px.bar(
    rank_spreads.sort_values(),
    orientation="h",
    title="Cross-Sectional Rank Spread by Feature",
    labels={"value": "Rank Spread", "index": "Feature"}
)

fig.show()

In [ ]:
diagnostics = pd.DataFrame({
    "rank_spread": rank_spreads,
    "signal_dispersion": avg_dispersion
})

fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text=diagnostics.index,
    title="Feature Diagnostic Map"
)

fig.update_traces(
    textposition="top center",
    textfont=dict(size=7)
)

fig.show()

In [ ]:
top_features = diagnostics.nlargest(10, "rank_spread").index

diagnostics["label"] = diagnostics.index.where(
    diagnostics.index.isin(top_features),
    ""
)

fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text="label",
    title="Feature Diagnostic Map"
)

fig.update_traces(textposition="top center", textfont=dict(size=9))

fig.add_vline(x=diagnostics.rank_spread.median(), line_dash="dash")
fig.add_hline(y=diagnostics.signal_dispersion.median(), line_dash="dash")

fig.show()

In [ ]:
def classify_feature(name):

    if "VEL" in name or "ACC" in name:
        return "MOM_DERIVATIVES"

    if "VOV" in name:
        return "VOL_OF_VOL"

    if "ALIGN" in name or "RATIO" in name or "EXP" in name:
        return "MOM_ALIGN"

    if "MSI" in name or "RATIO" in name or "EXP" in name:
        return "MSI"

    if "_Z" in name or "_P" in name:
        return "MOM_NORMALIZED"

    if name.startswith("MOM_"):
        return "MOM_LEVEL"

    return "OTHER"


diagnostics["feature_type"] = diagnostics.index.map(classify_feature)

In [ ]:
fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    color="feature_type",
    hover_name=diagnostics.index,
    title="Feature Diagnostic Map",
    width=900,
    height=600
)

top_features = diagnostics.nlargest(8, "rank_spread").index

diagnostics["label"] = diagnostics.index.where(
    diagnostics.index.isin(top_features),
    ""
)

fig.update_traces(
    text=diagnostics["label"],
    textposition="top center",
    textfont=dict(size=9)
)

fig.add_vline(
    x=diagnostics.rank_spread.median(),
    line_dash="dash"
)

fig.add_hline(
    y=diagnostics.signal_dispersion.median(),
    line_dash="dash"
)

fig.show()

### 6.2.3 Feature Correlation Analysis

In this section we analyze the **correlation between features** to detect:

- **Signal redundancy**
- **Structural clusters of indicators**
- **Effective dimensionality of the feature space**

The correlation between two features $f_i$ and $f_j$ is defined as:

$$
\rho_{ij} =
\frac{\text{Cov}(f_i, f_j)}
{\sigma_i \sigma_j}
$$

where:

- $\text{Cov}(f_i,f_j)$ is the covariance between the two signals  
- $\sigma_i, \sigma_j$ are their standard deviations

Interpretation:

| Correlation | Interpretation |
|---|---|
| $ \rho \approx 1 $ | highly redundant signals |
| $ \rho \approx 0 $ | independent signals |
| $ \rho \approx -1 $ | inversely related signals |

In **quantitative feature engineering** we aim for:

- a **diverse set of signals**
- **low redundancy**

To compute correlations we first transform the feature panel into a format where:

- each **row** represents an observation $(date, asset)$
- each **column** represents a **feature**

We then compute the **feature correlation matrix** and visualize it using a **heatmap**.

This analysis typically reveals **natural clusters of indicators**, for example:

- volatility level
- normalized volatility
- volatility term structure
- volatility derivatives
- volatility-of-volatility

These clusters help us understand the **structure of the feature space** and identify indicators that provide **genuinely new information**, as opposed to those that are simply **variations of the same signal**.

In [ ]:
# ==========================================
# 6.2.3 FEATURE CORRELATION ANALYSIS
# ==========================================

print("\n===== FEATURE CORRELATION ANALYSIS =====")

panel = features_panel

# ------------------------------------------
# Convert panel to long format
# (date, asset) x feature
# ------------------------------------------

panel_long = panel.stack(level="asset")

print("Panel long shape:", panel_long.shape)

# ------------------------------------------
# Compute correlation matrix
# ------------------------------------------

feature_corr = panel_long.corr()

print("Correlation matrix shape:", feature_corr.shape)

# ------------------------------------------
# Plot heatmap
# ------------------------------------------

import plotly.express as px

fig = px.imshow(
    feature_corr,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Feature Correlation Matrix"
)

fig.update_layout(
    width=900,
    height=900
)

fig.show()

### 6.2.4 Hierarchical Feature Clustering

While the raw correlation matrix provides useful information, it is often difficult to visually identify **groups of related features** when the matrix is ordered arbitrarily.

To better reveal the structure of the feature space, we apply **hierarchical clustering** to the correlation matrix.

The idea is to group features based on their **similarity**, measured through their pairwise correlations.

A common transformation is to convert correlation into a **distance metric**:

$$
d_{ij} = 1 - |\rho_{ij}|
$$

where:

- $\rho_{ij}$ is the correlation between features $i$ and $j$
- $d_{ij}$ represents the distance between the two signals

This distance metric ensures that:

- highly correlated features have **small distances**
- weakly correlated features have **larger distances**

Using this distance matrix, hierarchical clustering builds a **tree structure (dendrogram)** that groups similar features together.

The reordered correlation matrix reveals **natural clusters of signals**, which often correspond to feature families such as:

- volatility level indicators
- normalized volatility signals
- volatility term structure features
- volatility derivatives
- volatility-of-volatility measures

This step is extremely useful for:

- detecting **redundant signals**
- identifying **feature families**
- guiding **feature selection**
- reducing the **effective dimensionality** of the feature space

In [ ]:
# ==========================================
# 6.2.4 HIERARCHICAL FEATURE CLUSTERING
# ==========================================

print("\n===== HIERARCHICAL FEATURE CLUSTERING =====")

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import squareform

# ------------------------------------------
# Convert correlation to distance matrix
# ------------------------------------------

distance_matrix = 1 - np.abs(feature_corr)

# linkage requires condensed distance matrix
condensed_dist = squareform(distance_matrix)

linkage_matrix = linkage(condensed_dist, method="average")

# ------------------------------------------
# Clustered heatmap
# ------------------------------------------

sns.clustermap(
    feature_corr,
    row_linkage=linkage_matrix,
    col_linkage=linkage_matrix,
    cmap="RdBu_r",
    center=0,
    figsize=(12,12)
)

plt.title("Hierarchically Clustered Feature Correlation Matrix")
plt.show()

### 6.2.5 Feature Redundancy Detection

In large feature spaces, some indicators may carry **almost identical information**, i.e., they are **highly correlated** with each other.

Detecting these redundant features helps:

- **Reduce dimensionality**
- **Avoid multicollinearity**
- **Simplify feature selection**
- **Focus on truly informative signals**

We define redundancy as:

$$
|\rho_{ij}| > \rho_{threshold}
$$

where $\rho_{ij}$ is the correlation between features $i$ and $j$, and $\rho_{threshold}$ is typically set to 0.95–0.99.

This process outputs:

- a **list of highly correlated feature pairs**
- optionally, a **suggestion of features to drop** to remove redundancy

By systematically removing or combining redundant features, the **effective dimensionality** of the feature space is reduced without significant loss of information.

In [ ]:
# ==========================================
# 6.2.5 Feature Redundancy Detection
# ==========================================

print("\n===== FEATURE REDUNDANCY DETECTION =====")

# Threshold for considering features redundant
redundancy_threshold = 0.95

# Compute absolute correlation matrix
abs_corr = feature_corr.abs()

# Upper triangle mask to avoid duplicate pairs
upper_tri = np.triu(np.ones_like(abs_corr, dtype=bool), k=1)

# Select pairs with correlation above threshold
redundant_pairs = [
    (abs_corr.index[i], abs_corr.columns[j], abs_corr.iloc[i, j])
    for i in range(abs_corr.shape[0])
    for j in range(abs_corr.shape[1])
    if upper_tri[i, j] and abs_corr.iloc[i, j] > redundancy_threshold
]

# Convert to DataFrame for inspection
redundant_df = pd.DataFrame(redundant_pairs, columns=["Feature_1", "Feature_2", "Correlation"])
redundant_df = redundant_df.sort_values(by="Correlation", ascending=False).reset_index(drop=True)

print(f"Number of highly correlated feature pairs (>|{redundancy_threshold}|): {len(redundant_df)}")
display(redundant_df.head(20))  # show top 20 for quick inspection

### 6.2.6 Cross-Sectional Distribution Diagnostics

Analyzing the **cross-sectional distribution** of features at each time step helps us identify:

- Outliers in the feature space
- Skewness or heavy tails
- Differences in dispersion across features
- Features that may require normalization or winsorization before modeling

For a given feature \(f\) and time \(t\), let \(x_{t,i}^f\) denote the value for asset \(i\). We can compute:

1. **Cross-sectional mean**:
$$
\mu_t^f = \frac{1}{N} \sum_{i=1}^{N} x_{t,i}^f
$$

2. **Cross-sectional standard deviation**:
$$
\sigma_t^f = \sqrt{\frac{1}{N} \sum_{i=1}^{N} \left(x_{t,i}^f - \mu_t^f \right)^2}
$$

3. **Cross-sectional skewness** (optional):
$$
\text{skew}_t^f = \frac{\frac{1}{N} \sum_i (x_{t,i}^f - \mu_t^f)^3}{(\sigma_t^f)^3}
$$

We summarize the distribution **over time** by computing:

- Average cross-sectional mean
- Average cross-sectional standard deviation
- Extreme values or outliers

This diagnostic helps **compare features** and ensures that signals are **well-behaved across assets**, ready for factor modeling or cross-asset allocation.

In [ ]:
# ==========================================
# 6.2.6 Cross-Sectional Distribution Diagnostics (with Skewness)
# ==========================================

import pandas as pd
import plotly.express as px

print("\n===== CROSS-SECTIONAL DISTRIBUTION DIAGNOSTICS WITH SKEWNESS =====")

panel = features_panel  # wide panel: dates x (feature x asset)
panel_t = panel.T       # (feature, asset) x dates

features = panel_t.index.get_level_values("feature").unique()

# Crear DataFrame para stats
cs_distribution = pd.DataFrame(index=features, columns=[
    "cross_sectional_mean",
    "cross_sectional_std",
    "cross_sectional_skew",
    "cross_sectional_max",
    "cross_sectional_min"
], dtype=float)

# Calcular estadísticos
for f in features:
    feat_df = panel_t.xs(f)  # fechas x assets
    mu = feat_df.mean(axis=1)
    sigma = feat_df.std(axis=1)
    cs_distribution.loc[f, "cross_sectional_mean"] = mu.mean()
    cs_distribution.loc[f, "cross_sectional_std"] = sigma.mean()
    cs_distribution.loc[f, "cross_sectional_skew"] = ((feat_df.sub(mu, axis=0)**3).mean(axis=1) / sigma**3).mean()
    cs_distribution.loc[f, "cross_sectional_max"] = feat_df.max(axis=1).max()
    cs_distribution.loc[f, "cross_sectional_min"] = feat_df.min(axis=1).min()

cs_distribution = cs_distribution.sort_values(by="cross_sectional_std", ascending=False)

print("Cross-sectional distribution summary (with skewness):")
display(cs_distribution)

# Plot histogram para features seleccionadas
sample_features = cs_distribution.index[:8]  # top 8 por std

for feat in sample_features:
    fig = px.histogram(
        panel[feat],
        nbins=20,
        title=f"Cross-Sectional Distribution: {feat}"
    )
    fig.show()

# 7. Visualization

In [ ]:
# ============================================================
# 7. VISUALIZATION / EXPLORATORY PLOTS
# ============================================================

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 7.1 Configuration for plots
# ------------------------------------------------------------
SAMPLE_ASSETS = ["BTC", "SPY"]          # sample assets to inspect
SAMPLE_FEATURES = ["MOM_252_Z", "MOM_126_Z", "MOM_21_Z", "MOM_252"]  # features to inspect
TEXT_POSITION = "top center"
FIGURE_WIDTH = 1000
FIGURE_HEIGHT = 600

# ------------------------------------------------------------
# 7.2 Single Feature Across All Assets (with Prices)
# ------------------------------------------------------------
for feature in SAMPLE_FEATURES:
    df_feature = features_panel.xs(feature, level="feature", axis=1)
    
    # Crear figura con doble eje
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Agregar features
    for asset in df_feature.columns:
        fig.add_trace(
            go.Scatter(
                x=df_feature.index,
                y=df_feature[asset],
                mode="lines",
                name=f"{asset} Feature"
            ),
            secondary_y=False
        )
    
    # Agregar precios de los activos en eje secundario
    for asset in SAMPLE_ASSETS:
        fig.add_trace(
            go.Scatter(
                x=df_prices.index,
                y=df_prices[asset],
                mode="lines",
                line=dict(dash="dot"),
                name=f"{asset} Price"
            ),
            secondary_y=True
        )
    
    # Layout
    fig.update_layout(
        title=f"{feature} Across Assets with Prices",
        xaxis_title="Date",
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT
    )
    fig.update_yaxes(title_text="Feature Value", secondary_y=False)
    fig.update_yaxes(title_text="Price", secondary_y=True)
    
    fig.show()

# ------------------------------------------------------------
# 7.3 Single Asset Across Multiple Features (with Prices)
# ------------------------------------------------------------
for asset in SAMPLE_ASSETS:
    df_asset = features_panel.xs(asset, level="asset", axis=1)
    df_asset = df_asset[SAMPLE_FEATURES]
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Features
    for feature in df_asset.columns:
        fig.add_trace(
            go.Scatter(
                x=df_asset.index,
                y=df_asset[feature],
                mode="lines",
                name=f"{feature}"
            ),
            secondary_y=False
        )
    
    # Prices
    fig.add_trace(
        go.Scatter(
            x=df_prices.index,
            y=df_prices[asset],
            mode="lines",
            line=dict(dash="dot"),
            name=f"{asset} Price"
        ),
        secondary_y=True
    )
    
    fig.update_layout(
        title=f"Selected Features for {asset} with Price",
        xaxis_title="Date",
        width=FIGURE_WIDTH,
        height=FIGURE_HEIGHT
    )
    fig.update_yaxes(title_text="Feature Value", secondary_y=False)
    fig.update_yaxes(title_text="Price", secondary_y=True)
    
    fig.show()

# ------------------------------------------------------------
# 7.4 Feature Diagnostic Map (Scatter)
# ------------------------------------------------------------
# Asegurate de tener 'diagnostics' dataframe con columnas 'rank_spread' y 'signal_dispersion'
fig = px.scatter(
    diagnostics,
    x="rank_spread",
    y="signal_dispersion",
    text=diagnostics.index,
    title="Feature Diagnostic Map"
)
fig.update_traces(textposition=TEXT_POSITION)
fig.update_layout(width=FIGURE_WIDTH, height=FIGURE_HEIGHT)
fig.show()

# ------------------------------------------------------------
# 7.5 Heatmap of Feature Correlations
# ------------------------------------------------------------
corr_matrix = features_panel.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.show()

# ------------------------------------------------------------
# 7.6 Outlier Inspection for a Feature (with Prices)
# ------------------------------------------------------------
feature_to_inspect = "MOM_252_Z"
df_feature = features_panel.xs(feature_to_inspect, level="feature", axis=1)
mean_series = df_feature.mean()
std_series = df_feature.std()
upper_bound = mean_series + 3 * std_series
lower_bound = mean_series - 3 * std_series

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Features
for asset in df_feature.columns:
    fig.add_trace(
        go.Scatter(
            x=df_feature.index,
            y=df_feature[asset],
            mode="lines",
            name=f"{asset} Feature"
        ),
        secondary_y=False
    )
    
    # Outliers
    outliers = df_feature[asset][(df_feature[asset] > upper_bound[asset]) | (df_feature[asset] < lower_bound[asset])]
    fig.add_trace(
        go.Scatter(
            x=outliers.index,
            y=outliers.values,
            mode="markers",
            marker=dict(color="red", size=8),
            name=f"{asset} outlier"
        ),
        secondary_y=False
    )

# Prices
for asset in SAMPLE_ASSETS:
    fig.add_trace(
        go.Scatter(
            x=df_prices.index,
            y=df_prices[asset],
            mode="lines",
            line=dict(dash="dot"),
            name=f"{asset} Price"
        ),
        secondary_y=True
    )

fig.update_layout(
    title=f"Outlier Inspection for {feature_to_inspect} with Prices",
    xaxis_title="Date",
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT
)
fig.update_yaxes(title_text="Feature Value", secondary_y=False)
fig.update_yaxes(title_text="Price", secondary_y=True)

fig.show()

# 8. Exports

In [ ]:
# -----------------------------------------
# 8.1 Final Feature Panel
# -----------------------------------------

print("Feature panel shape:", features_panel.shape)
print("Column levels:", features_panel.columns.names)

features_panel.head()

In [ ]:
# -----------------------------------------
# Panel metadata
# -----------------------------------------

n_dates = features_panel.shape[0]
n_features = features_panel.columns.get_level_values("feature").nunique()
n_assets = features_panel.columns.get_level_values("asset").nunique()

print("Dates:", n_dates)
print("Features:", n_features)
print("Assets:", n_assets)

In [ ]:
# ============================================================
# 8. EXPORT / SAVE FEATURES
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# 8.1 Helper: detect project root
# ------------------------------------------------------------
def find_project_root(marker_folder="data"):
    current_path = os.path.abspath(os.getcwd())
    while True:
        if marker_folder in os.listdir(current_path):
            return current_path
        parent = os.path.dirname(current_path)
        if parent == current_path:
            raise RuntimeError(f"No '{marker_folder}' folder found in directory tree.")
        current_path = parent

project_root = find_project_root("data")
print("Project root detected:", project_root)

# ------------------------------------------------------------
# 8.2 Feature family configuration
# ------------------------------------------------------------
feature_family = FEATURE_FAMILY  # e.g., "volatility", "momentum"
base_feature_dir = os.path.join(project_root, "data", "features", "asset", feature_family)
os.makedirs(base_feature_dir, exist_ok=True)

# ------------------------------------------------------------
# 8.3 Clean and sort panel
# ------------------------------------------------------------
features_panel = features_panel.sort_index()         # sort rows by date
features_panel = features_panel.sort_index(axis=1)  # sort MultiIndex columns

# ------------------------------------------------------------
# 8.4 Export per asset
# ------------------------------------------------------------
assets = features_panel.columns.get_level_values("asset").unique()

for asset in assets:
    df_asset = features_panel.xs(asset, level="asset", axis=1)
    df_asset.columns.name = None  # clean column metadata
    filename = os.path.join(base_feature_dir, f"{asset}.parquet")
    df_asset.to_parquet(filename, compression="zstd")
    print(f"{feature_family} exported for {asset} -> {filename}")